### Multimodal RAG (PDF With Images)



In [10]:
import fitz   # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain_classic.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.schema.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [12]:
###Clip Model
import os
from dotenv import load_dotenv
load_dotenv()

## set up the environment
from transformers import CLIPProcessor, CLIPModel

### initialize the Clip Model for unified embeddings
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 20127.48it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [25]:
## Embedding function
def embed_image(image):
    inputs = clip_processor(
        images=image,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = clip_model(**inputs)
        features = outputs.image_embeds

        # Normalize embeddings
        features = features / features.norm(
            p=2, dim=-1, keepdim=True
        )

    return features.squeeze(0).cpu().numpy()


def embed_text(text):
    inputs = clip_processor(
        text=[text],
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)

        # Handle different Transformers versions
        if not isinstance(features, torch.Tensor):
            if hasattr(features, "text_embeds"):
                features = features.text_embeds
            elif hasattr(features, "pooler_output"):
                features = features.pooler_output
            else:
                raise TypeError(
                    f"Unexpected CLIP output type: {type(features)}"
                )

        features = features / features.norm(
            p=2, dim=-1, keepdim=True
        )

    return features.squeeze(0).cpu().numpy()

In [26]:
## Process PDf
pdf_path = "multimodal_sample.pdf"
doc= fitz.open(pdf_path)
# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {} # store actual image data for llm

## Text splitter 
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [27]:
doc

Document('multimodal_sample.pdf')

In [28]:
for i,page in enumerate(doc):
    ## process text
    text=page.get_text()
    if text.strip():
        ##create temporary document for splitting
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        #Embed each chunk using CLIP
        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)



    ## process images
    ##Three Important Actions:

    ##Convert PDF image to PIL format
    ##Store as base64 for GPT-4V (which needs base64 images)
    ##Create CLIP embedding for retrieval

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Convert to PIL Image
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            # Create unique identifier
            image_id = f"page_{i}_img_{img_index}"
            
            # Store image as base64 for later use with GPT-4V
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            # Embed image using CLIP
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)
            
            # Create document for image
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()


Error processing image 0 on page 0: You have to specify input_ids


In [29]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.')]

In [30]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[-2.67245667e-03,  1.28300050e-02, -5.18314056e-02,
         4.14879918e-02, -2.33942103e-02, -7.55866664e-03,
        -3.67659554e-02,  1.19710706e-01,  8.52080882e-02,
         2.05425546e-03, -1.11534800e-02, -1.29592447e-02,
         5.25014438e-02, -3.65394820e-03,  4.76078652e-02,
         1.58372559e-02,  2.03388408e-02,  4.35362309e-02,
        -3.29167582e-03,  2.03181412e-02,  1.88024528e-03,
        -4.23494130e-02,  5.44102723e-03,  3.70935984e-02,
        -1.65623128e-02,  6.48644054e-03, -4.78012003e-02,
         8.67483951e-03,  5.88859692e-02, -3.21393870e-02,
         4.32439633e-02,  9.65303835e-03, -4.47918894e-03,
        -1.94857772e-02, -3.63502875e-02, -1.23471543e-02,
        -2.17929576e-02, -1.99016202e-02,  8.09619799e-02,
        -3.32986340e-02, -2.38901321e-02, -3.96138988e-02,
        -1.27279637e-02,  3.50380577e-02, -2.52217464e-02,
         2.00030091e-03,  1.49660325e-02, -2.31976453e-02,
        -6.86791539e-02, -5.25775074e-04, -2.22545490e-0

In [31]:
(all_docs,embeddings_array)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.')],
 array([[-2.67245667e-03,  1.28300050e-02, -5.18314056e-02,
          4.14879918e-02, -2.33942103e-02, -7.55866664e-03,
         -3.67659554e-02,  1.19710706e-01,  8.52080882e-02,
          2.05425546e-03, -1.11534800e-02, -1.29592447e-02,
          5.25014438e-02, -3.65394820e-03,  4.76078652e-02,
          1.58372559e-02,  2.03388408e-02,  4.35362309e-02,
         -3.29167582e-03,  2.03181412e-02,  1.88024528e-03,
         -4.23494130e-02,  5.44102723e-03,  3.70935984e-02,
         -1.65623128e-02,  6.48644054e-03, -4.78012003e-02,
          8.67483951e-03

In [32]:
# Create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,  # We're using precomputed embeddings
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [38]:
# Initialize GPT-4 Vision model
llm = init_chat_model("groq:openai/gpt-oss-120b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B11863F620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B11AE34980>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [39]:
def retrieve_multimodal(query, k=5):
    """Unified retrieval using CLIP embeddings for both text and images."""
    # Embed query using CLIP
    query_embedding = embed_text(query)
    
    # Search in unified vector store
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    
    return results

In [40]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []
    
    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [41]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V
    response = llm.invoke([message])
    
    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response.content

In [42]:
if __name__ == "__main__":
    # Example queries
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 1 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...


Answer: The chart (the one referenced on the first page of the document) illustrates a clear upward trajectory in revenue across the three quarters:

* **Q1** – Revenue shows a modest rise, reflecting the introduction of new product lines.  
* **Q2** – The increase is larger than in Q1, driven by the impact of marketing campaigns.  
* **Q3** – Revenue spikes dramatically, indicating “exponential” growth that stems from the company’s global expansion efforts.

Overall, the chart demonstrates that revenue grew steadily throughout the year, with the most pronounced growth occurring in Q3.

Query: Summarize the main findings from the document
--------------------------------------------------

Retrieved 1 documents:
  - Text f